In [0]:
-- Concession lines for active CineClub members, P12M, Cineplex theatres
CREATE OR REPLACE TEMP VIEW cc_popcorn_lines AS
WITH active_members AS (
  SELECT DISTINCT cde_id AS CDE_ID
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
  WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                            FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
    AND renewal_status LIKE 'Active%'
    AND recognitions_left < 63          -- BR-025 test-account ceiling
    AND cde_id > 0
)
SELECT t.CDE_ID, t.VisitDateId, t.LocationId, t.Quantity,
       UPPER(TRIM(i.Item_Description)) AS item_desc
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN active_members m                                    ON t.CDE_ID = m.CDE_ID
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON t.LocationId = l.LocationID
JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i     ON t.ItemSkey = i.itemskey
WHERE t.ConcessionFLAG = 'Yes'
  AND l.LC_Location_Type_Description = 'Cineplex Theatre'
  AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No'
  AND t.VisitDateId BETWEEN 20251029 AND 20261028;

-- One row per popcorn visit (member x date x location) with add-on flags
CREATE OR REPLACE TEMP VIEW cc_popcorn_visits AS
WITH line_flags AS (
  SELECT CDE_ID, VisitDateId, LocationId,
         -- popcorn: Wrapped V5 cell 11 (sizes + upsizes)
         (item_desc LIKE '%POPCORN%LARGE%'      OR item_desc LIKE '%LARGE%POPCORN%'
          OR item_desc LIKE '%POPCORN%REGULAR%' OR item_desc LIKE '%REGULAR%POPCORN%'
          OR item_desc LIKE '%POPCORN%SMALL%'   OR item_desc LIKE '%SMALL%POPCORN%'
          OR item_desc IN ('VHO CORN - UPSIZE TO REGULAR CORN',
                           'VHO CORN - UPSIZE TO LARGE CORN',
                           'VHO CORN - UPSIZE SMALL TO LARGE CORN'))   AS is_popcorn,
         -- butter add-ons: layered vs on top (on top = Butter on Top + non-layered VHO butter)
         CASE WHEN (item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' OR item_desc LIKE 'LAYERED BUTTER%')
                   AND item_desc LIKE '%LAYER%'                     THEN 'LAYERED'
              WHEN item_desc = 'BUTTER ON TOP'
                   OR item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' THEN 'ON_TOP'
         END                                                           AS butter_type,
         -- shake-it add-on
         item_desc LIKE '%SHAKE IT%'                                   AS is_shake
  FROM cc_popcorn_lines
)
SELECT CDE_ID, VisitDateId, LocationId,
       MAX(butter_type IS NOT NULL)    AS has_butter,
       MAX(butter_type <=> 'LAYERED')  AS has_layered,
       MAX(butter_type <=> 'ON_TOP')   AS has_on_top,
       MAX(is_shake)                   AS has_shake
FROM line_flags
GROUP BY CDE_ID, VisitDateId, LocationId
HAVING MAX(is_popcorn);


WITH member_flags AS (
  SELECT CDE_ID,
         MAX(INT(has_butter))  AS has_butter,
         MAX(INT(has_layered)) AS has_layered,
         MAX(INT(has_on_top))  AS has_on_top,
         MAX(INT(has_shake))   AS has_shake
  FROM cc_popcorn_visits
  GROUP BY CDE_ID
)
SELECT 'Members' AS base, COUNT(*) AS popcorn_base,
       ROUND(100*AVG(has_butter),1)  AS pct_butter,
       ROUND(100*AVG(has_layered),1) AS pct_layered_butter,
       ROUND(100*AVG(has_on_top),1)  AS pct_butter_on_top,
       ROUND(100*AVG(has_shake),1)   AS pct_shake_its
FROM member_flags
UNION ALL
SELECT 'Visits', COUNT(*),
       ROUND(100*AVG(INT(has_butter)),1),
       ROUND(100*AVG(INT(has_layered)),1),
       ROUND(100*AVG(INT(has_on_top)),1),
       ROUND(100*AVG(INT(has_shake)),1)
FROM cc_popcorn_visits;

SELECT CASE WHEN item_desc = 'BUTTER ON TOP' THEN 'Butter on top'
            WHEN (item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' OR item_desc LIKE 'LAYERED BUTTER%')
                 AND item_desc LIKE '%LAYER%' THEN 'Layered butter'
            WHEN item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' THEN 'Regular butter'
            WHEN item_desc LIKE '%SHAKE IT%' THEN 'Shake-it'
            WHEN item_desc LIKE '%POPCORN%LARGE%'      OR item_desc LIKE '%LARGE%POPCORN%'
              OR item_desc LIKE '%POPCORN%REGULAR%' OR item_desc LIKE '%REGULAR%POPCORN%'
              OR item_desc LIKE '%POPCORN%SMALL%'   OR item_desc LIKE '%SMALL%POPCORN%'
              OR item_desc IN ('VHO CORN - UPSIZE TO REGULAR CORN',
                               'VHO CORN - UPSIZE TO LARGE CORN',
                               'VHO CORN - UPSIZE SMALL TO LARGE CORN') THEN 'Popcorn'
       END AS bucket,
       item_desc, COUNT(*) AS lines, SUM(Quantity) AS qty
FROM cc_popcorn_lines
GROUP BY 1, 2
HAVING bucket IS NOT NULL
ORDER BY bucket, qty DESC;


-- WITH lines AS (
--   SELECT UPPER(TRIM(i.Item_Description)) AS item_desc, t.Quantity, t.Net_Revenue
--   FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
--   JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i     ON t.ItemSkey = i.itemskey
--   JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l ON t.LocationId = l.LocationID
--   WHERE t.ConcessionFLAG = 'Yes'
--     AND l.LC_Location_Type_Description = 'Cineplex Theatre'
--     AND t.STOREFLAG = 'No' AND t.RecRoomFLAG = 'No'
--     AND t.VisitDateId BETWEEN 20251029 AND 20261028
-- ),
-- bucketed AS (
--   SELECT CASE WHEN item_desc = 'BUTTER ON TOP' THEN 'Butter on top'
--               WHEN (item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' OR item_desc LIKE 'LAYERED BUTTER%')
--                    AND item_desc LIKE '%LAYER%' THEN 'Layered butter'
--               WHEN item_desc RLIKE '^VHO (CORN|DELIVERY) - BUTTER' THEN 'Regular butter'
--               WHEN item_desc LIKE '%SHAKE IT%' THEN 'Shake-it'
--               WHEN item_desc LIKE '%POPCORN%LARGE%'      OR item_desc LIKE '%LARGE%POPCORN%'
--                 OR item_desc LIKE '%POPCORN%REGULAR%' OR item_desc LIKE '%REGULAR%POPCORN%'
--                 OR item_desc LIKE '%POPCORN%SMALL%'   OR item_desc LIKE '%SMALL%POPCORN%'
--                 OR item_desc IN ('VHO CORN - UPSIZE TO REGULAR CORN',
--                                  'VHO CORN - UPSIZE TO LARGE CORN',
--                                  'VHO CORN - UPSIZE SMALL TO LARGE CORN') THEN 'Popcorn'
--          END AS bucket,
--          item_desc, Quantity, Net_Revenue
--   FROM lines
-- )
-- SELECT bucket, item_desc,
--        SUM(Quantity)                                                        AS qty,
--        ROUND(SUM(Net_Revenue), 2)                                           AS net_revenue,
--        ROUND(SUM(Net_Revenue) / NULLIF(SUM(Quantity), 0), 2)                AS avg_price_all,
--        ROUND(SUM(CASE WHEN Net_Revenue > 0 THEN Net_Revenue END)
--              / NULLIF(SUM(CASE WHEN Net_Revenue > 0 THEN Quantity END), 0), 2) AS avg_price_paid_only,
--        ROUND(100 * SUM(CASE WHEN Net_Revenue = 0 THEN Quantity END)
--              / NULLIF(SUM(Quantity), 0), 1)                                 AS pct_units_free
-- FROM bucketed
-- WHERE bucket IS NOT NULL
-- GROUP BY bucket, item_desc
-- ORDER BY bucket, qty DESC;